In [131]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

MARKET = "PT-reUSD-25JUN2026_usdc"


EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"

df = pd.read_csv(f"/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched/{MARKET}.csv")

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type,collateral_asset_symbol,loan_asset_symbol
0,0x728503f0fd942199a221fe10fa3c3fbda2d66aadc812...,MarketSupply,1765292819,0x000000000000000000000000000000000000dEaD,1000,0.001000,0,0,PT-reUSD-25JUN2026_usdc,2025-12-09 15:06:59,0x9bc98c2f20ac58287ef2c860eea53a2fdc27c17a7817...,0.000,0.0,0.001,0.000,0.0,0.0,1,0.009947,0.0,0.009947,0.000000,0.92221,0.999870,0.0,0.0,0.0,0.0,0.0,0.000,0.000000,0.000,0.001,0.000000,0.0,0.0000,loan_position_supply,False,0.0,0.0,0.0,0.0,0.0,0.0,loan_position_supply,PT-reUSD-25JUN2026,USDC
1,0x3bc73a282d7083cd6556a372937751823f49d96ec04f...,MarketSupplyCollateral,1765296899,0x201051Ae0FddaC0Ce47B299E4673cAA39f32A961,2000,0.001844,0,0,PT-reUSD-25JUN2026_usdc,2025-12-09 16:14:59,0x9bc98c2f20ac58287ef2c860eea53a2fdc27c17a7817...,0.001,0.0,0.001,0.001,0.0,1.0,2,0.009947,0.0,0.159153,0.159153,0.92221,0.999806,0.0,0.0,0.0,0.0,0.0,0.002,0.001844,0.001,0.000,0.542176,0.0,1.6883,position_open,False,0.0,0.0,0.0,0.0,0.0,0.0,position_open,PT-reUSD-25JUN2026,USDC


In [58]:
TOKEN_NAME = df["collateral_asset_symbol"].unique()[0]
yield_hist = pd.read_csv(f"/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/{TOKEN_NAME}.csv")
market_df = market_df.sort_values('timestamp')
yield_hist = yield_hist.sort_values('timestamp')
yield_hist = yield_hist[[
    "timestamp",
    "implied_apy",
    "underlying_apy",
    "base_apy",
]]
market_df = pd.merge_asof(market_df, yield_hist, on='timestamp', direction='nearest')
market_df.head(2)

# market_df['date'] = pd.to_datetime(market_df['datetime']).dt.date
# yield_hist['date'] = pd.to_datetime(yield_hist['date']).dt.date
# market_df = pd.merge(market_df, yield_hist[["date", "apy"]], on='date', how='left')


,timestamp,datetime,total_supply,total_borrow,utilization,borrow_rate,supply_rate,volatility_1h,drawdown_1h,volatility_6h,drawdown_6h,collateral_price,loan_asset_price,avg_health_factor,borrow_rate_rolling,supply_rate_rolling,asset_price,implied_apy,underlying_apy,base_apy
0,1765292400,2025-12-09 15:00:00,0.000,0.0,0.0,0.000000,0.0,0,0,0.0,0.0,0.00000,0.00000,0.0,0.000000,0.0,0.00000,0.162464,0.069637,0.115996
1,1765296000,2025-12-09 16:00:00,0.001,0.0,0.0,0.009947,0.0,0,0,0.0,0.0,0.92221,0.99987,0.0,0.004974,0.0,0.92221,0.159172,0.066772,0.116678


In [29]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
%autoreload 2
from utils import (
    create_hourly_user_dataset,
)
from visualization_utils import (
    plot_user_metrics,
)

In [50]:
supplyers = df[df["event_type"] == "MarketSupply"]["user_address"].unique()
df[(df["vault_flg"] == False) & (~df["user_address"].isin(supplyers))]["user_address"].value_counts().sample(30)

user_address
0x4Cb43bc2F2977D10069D05A7d3f75D99d373d8bB     91
0x826D0765E060127BD816Baa525Fd198e64d2bC0b     65
0x05f987caa2fEDdEbBA319eFC95D86F7Ab021A241     13
0xCa9ba74eE20917211ef646AC51ACcc287F27538b     13
0x79E4050bE416af6B94390162C91114d8bd568528     11
0x9353C8cDa3e7513961e0Fab3e722395DC127B454      4
0x129A803d1ac2A186759F8988eEb3e6A1C4C7Db18     78
0x16c55b25f1b55b7BE513D58f27b7Fa369CE95137    104
0xfA5b8f59074F1d908d0a4A0f88aA4CD0B6F120C6     14
0xE58a50b84063E04Feeb2DCDFE522B018a3eB16cD     91
0x4877589872595b6dae33aDc608173F38F631da9E     52
0xC509f65e91e35cD7c4Dc242c827fA6Ca7c35Da6B      4
0xf7f6B258aa09e3559678066c5FadeDa875580DF7     21
0xe1eCC49D4994f75aCfe3aD0f7643be427eaFB62B     65
0x1A924135Ef7e449Baf2a5AFD194C32F40a6a2a42     65
0xBd7F505D86106AaC3FEBb5Acb0c0DF5B6171F0B9     91
0x62A55f8a930c58B7e7fBDf8FBC56F481fD3C99d7     65
0x8f5855D30610b176c972F844B93A28c6D1553416      7
0x201051Ae0FddaC0Ce47B299E4673cAA39f32A961      3
0xf29ce940178C8794802fB48a6c1B2EdDdAC

In [75]:
df[(~df["user_address"].isin(supplyers))].groupby("user_address")["debt_after"].max().sort_values()[::-1][:30]

user_address
0xf991AcD672d5B344aFCB29931Af6401ed32e2691    1.340151e+07
0x614D98a57A5D879D717152dE0690ed2b04562adE    1.096428e+07
0x60753E8622a6CFEA1Df449A9625a9778BB922b72    7.445941e+06
0x3874b310aD937f556144Bb0f6EA06461Ac209CCA    3.202793e+06
0x029cd954C38838b2731Cb5618E5C950ed4766956    1.949806e+06
0x82fCFFFE80D80365A5F2f42a6ea87d1e815Bec4C    1.857341e+06
0x7F86D8B6aC42503646c3ECb9Ca4940332627A8e3    1.835892e+06
0x2424618d727E17A38cad7D531dfC90540DCDB566    1.524125e+06
0x22F34f08406073bd27239cBaCF7942Ba0477704A    1.468937e+06
0x5A98dBcEeCa053b7845Fc4dD79124F66908AdE50    1.028920e+06
0x04F461Af0894C871D3997B29EAc293260f68021C    7.397011e+05
0xe29300C00618DD16C9D332f899B71F4362Be6B1d    7.353218e+05
0xFeffC588f4eA9c18A8B6CfB96e2EA9556BC516Fd    7.322780e+05
0x7C9063122C01837fe83dA2521056e10C9b6dd129    7.197857e+05
0x0cA6CAdb6a8cA4199788b9a060C3285Fe7B897Fd    6.233200e+05
0x8e6ed79c48944265FE5D004010c11CB28aa105B4    5.698788e+05
0xFA1F2977a0C5dB2eB171c2881540A7ce068EC2CC 

In [137]:
df[df["user_address"] == addr]["hash"].unique()[0]
df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type,collateral_asset_symbol,loan_asset_symbol
0,0x728503f0fd942199a221fe10fa3c3fbda2d66aadc812...,MarketSupply,1765292819,0x000000000000000000000000000000000000dEaD,1000,0.001000,0,0,PT-reUSD-25JUN2026_usdc,2025-12-09 15:06:59,0x9bc98c2f20ac58287ef2c860eea53a2fdc27c17a7817...,0.000,0.0,0.001,0.000,0.0,0.0,1,0.009947,0.0,0.009947,0.000000,0.92221,0.999870,0.0,0.0,0.0,0.0,0.0,0.000,0.000000,0.000,0.001,0.000000,0.0,0.0000,loan_position_supply,False,0.0,0.0,0.0,0.0,0.0,0.0,loan_position_supply,PT-reUSD-25JUN2026,USDC
1,0x3bc73a282d7083cd6556a372937751823f49d96ec04f...,MarketSupplyCollateral,1765296899,0x201051Ae0FddaC0Ce47B299E4673cAA39f32A961,2000,0.001844,0,0,PT-reUSD-25JUN2026_usdc,2025-12-09 16:14:59,0x9bc98c2f20ac58287ef2c860eea53a2fdc27c17a7817...,0.001,0.0,0.001,0.001,0.0,1.0,2,0.009947,0.0,0.159153,0.159153,0.92221,0.999806,0.0,0.0,0.0,0.0,0.0,0.002,0.001844,0.001,0.000,0.542176,0.0,1.6883,position_open,False,0.0,0.0,0.0,0.0,0.0,0.0,position_open,PT-reUSD-25JUN2026,USDC


In [139]:
addr = "0xd7Fc4Ab828AFc1bb4b217f337f1777Ca856Efd12"
# df["hash"] = df["hash"].str[:10]
df[df["user_address"] == addr][[
    "datetime",
    "type",
    "assets",
    "debt_after",
    "collateral_value_after",
    "ltv_after",
    "event_type",
    "event_sequence_type",
    "collateral_price",
    
    "borrow_rate_before",
    "borrow_rate_after",
    
    # "hash"
]].sort_values("datetime")[:50]

,datetime,type,assets,debt_after,collateral_value_after,ltv_after,event_type,event_sequence_type,collateral_price,borrow_rate_before,borrow_rate_after
360,2025-12-21 22:43:59,MarketBorrow,265000000000,265037.533525,319562.524636,0.829376,position_open,position_open,0.950349,0.07118,0.071953
361,2025-12-21 22:43:59,MarketSupplyCollateral,336257909551,265037.533525,319562.524636,0.829376,position_open,position_open,0.950349,0.07118,0.071953


In [59]:
hourly_data = create_hourly_user_dataset(
    df,
    market_df,
    n_hours=1,
    threshold_date="2026-03-09",
    additional_market_df_metrics=["implied_apy", "base_apy", "underlying_apy"],
    stop_at_close=False
)


In [117]:
plot_user_metrics(hourly_data, ['underlying_apy', 'collateral_price'], user_address="0x817a430658772207699a3b88F629876fdb9debF1", lookback_hours=None,
                  y_ranges={'borrow_rate': [0,0.2], 'underlying_apy': [0, 0.2]}
                  )

In [113]:
tmp = df[(~df["user_address"].isin(supplyers))].groupby("user_address")["debt_after"].max().sort_values()[::-1]
tmp = tmp[(0 < tmp)& (tmp < 100_000)]
len(tmp), df["user_address"].nunique()
tmp[:10]

user_address
0xE16BE042f9433779909a972669bE3A2003956348    98889.391620
0x97E6A34897a32e7103F3CF260f0C9cA5cA1fB90B    97244.954782
0x743D7B30661d65B41960Bf6b5d1bB93CF7972A73    88486.474959
0x259F5bfEAb72c188618F11155108bc4666261057    64903.244089
0x77b66C6f4C26056c0FE2604c45c465b13739e3da    53988.400575
0x6641E5F28a5b75823ebb81F7D22d23CD10E33d32    32560.567139
0x95D2602d30DA1179fd13274839e60345857ca648    31342.648595
0x484280E12F97905D2883A71f1596E1BC650e4809    18576.508155
0xbc0Ca340958b32F478429b3ACD4dDe39d45aD467    13964.695235
0x307538d1fF324E77E5eB202e5Aca37E0BAd359d6    13436.510124
Name: debt_after, dtype: float64

In [141]:
tmp = df[(~df["user_address"].isin(supplyers))].groupby("user_address")["debt_after"].max().sort_values()[::-1]
tmp = tmp[(0 < tmp)& (tmp < 1_000_000)]
len(tmp), df["user_address"].nunique()
for ad in tmp[:15].index:
    plot_user_metrics(hourly_data, ['implied_apy', 'borrow_rate',], user_address=ad, lookback_hours=None,
                      y_ranges={'ltv': [0.5,1],'borrow_rate': [0,0.2], 'underlying_apy': [0, 0.2], 'implied_apy': [0, 0.2], }
                      )


    User 0x04F461Af0894C871D3997B29EAc293260f68021C
    Max debt 739701.1139774928
    Max LTV 0.7691988822143927
    Position opens 1
    



    User 0xe29300C00618DD16C9D332f899B71F4362Be6B1d
    Max debt 735321.8436731098
    Max LTV 0.8256024637565181
    Position opens 1
    



    User 0xFeffC588f4eA9c18A8B6CfB96e2EA9556BC516Fd
    Max debt 732278.0338714857
    Max LTV 0.8210910280977847
    Position opens 1
    



    User 0x7C9063122C01837fe83dA2521056e10C9b6dd129
    Max debt 719785.6870731824
    Max LTV 0.9080325768980253
    Position opens 3
    



    User 0x0cA6CAdb6a8cA4199788b9a060C3285Fe7B897Fd
    Max debt 623319.9780107194
    Max LTV 0.8838695319348856
    Position opens 1
    


...


    User 0x6733F396C7d538F1CfafF27d2AC9Cb35dD30C442
    Max debt 268605.442138881
    Max LTV 0.8489255378017547
    Position opens 1
    



    User 0xd7Fc4Ab828AFc1bb4b217f337f1777Ca856Efd12
    Max debt 265037.5335249631
    Max LTV 0.8575225147136812
    Position opens 1
    



    User 0xE588A8AcC5baa6F1510a6c089501426b3861BcA9
    Max debt 251973.83370852697
    Max LTV 0.8583532655312842
    Position opens 1
    



    User 0x594261e60FBab0DA814075Bf752d5E044A27269C
    Max debt 248168.7769626613
    Max LTV 0.886942387746968
    Position opens 1
    



    User 0x9082126715CAd42793C89B5bd60691834cbb7900
    Max debt 242958.36013723133
    Max LTV 0.8849827418522649
    Position opens 1
    
